# Evaluation notebook
This notebook evaluates a dense embedding baseline using the same binary relevance setting as the existing TF-IDF, Qwen, and MiniLM notebooks.


In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sentence_transformers import SentenceTransformer

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sentence_transformer,
    evaluate_tfidf,
    load_triplets,
)

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_dbpedia.json")
TOPKS = [3, 5, 10, 20]
BATCH = 32

E5_MODEL_NAME = "intfloat/e5-base-v2"

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")

Loaded 337 items
Global corpus size: 1677
Eval anchors with non-empty positives: 337


In [3]:
evaluate_sentence_transformer(
    E5_MODEL_NAME,
    "Baseline: E5",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    batch_size=BATCH,
    transform=lambda s: f"query: {s}",
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4998.33it/s]
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Baseline: E5
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0297  0.0134  0.0162  0.0554  0.0831    0.0297  0.0309
@5     0.0237  0.0176  0.0170  0.0587  0.0979    0.0237  0.0290
@10    0.0208  0.0303  0.0205  0.0633  0.1306    0.0208  0.0321
@20    0.0209  0.0527  0.0251  0.0682  0.2077    0.0209  0.0409
